# Data processing for Feature Pipeline

In [3]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType
from utils.helper import generate_first_of_month_dates
import utils.data_processing_bronze_table
import utils.data_processing_silver_table

## Set up & config

In [4]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/29 01:11:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
# Setup config
start_date_str = "2023-01-01"
label_end_date_str = "2024-12-01"
snapshot_date_str = "2023-01-01"
feature_end_date_str = "2025-01-01"

In [9]:
# Generate date string list
date_str_lst = generate_first_of_month_dates(start_date_str, feature_end_date_str)

## Build bronze tables

In [73]:
# Run bronze backfill
feature_sources = {
    "attributes": "data/features_attributes.csv",
    "financials": "data/features_financials.csv",
    "clickstream": "data/feature_clickstream.csv"
}

for table_name, source_path in feature_sources.items():
    bronze_directory = f"datamart/bronze/{table_name}"
    for date in date_str_lst:
        utils.data_processing_bronze_table.process_feature_bronze_table(
            date,
            source_path,
            table_name,
            bronze_directory,
            spark
        )

2023-01-01 row count: 530
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_01_01.csv
2023-02-01 row count: 501
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_02_01.csv
2023-03-01 row count: 506
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_03_01.csv
2023-04-01 row count: 510
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_04_01.csv
2023-05-01 row count: 521
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_05_01.csv
2023-06-01 row count: 517
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_06_01.csv
2023-07-01 row count: 471
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_07_01.csv
2023-08-01 row count: 481
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_08_01.csv
2023-09-01 row count: 454
saved to: datamart/bronze/attributes/bronze_feature_attributes_2023_09_01.csv
2023-10-01 row count: 487
saved to: datamart/bronze/attributes/b

## EDA & Silver preparation for `features_attributes` data

**Goal**: Use EDA to justify the cleaning rules, then build a clean Silver dataframe.

- Keep `customer_id` as the entity/join key;
- Clean `age`;
- Clean `occupation`;
- Keep and validate `snapshot_date` for temporal alignment;
- Exclude `Name` and `SSN` from Silver because they are PII and are not required for downstream feature generation. The raw values remain available in Bronze.


### 1. Exploration at file-level

Confirm that the Bronze partitions have the expected structure and that `Customer_ID` is unique within each snapshot.

In [74]:
# Explore data at file-level
attributes_dir = "datamart/bronze/attributes"
attribute_files = sorted(glob.glob(os.path.join(attributes_dir, "*.csv")))

results = []

for file_path in attribute_files:
    file_name = os.path.basename(file_path)

    # Expected Bronze naming: bronze_attributes_YYYY_MM_DD.csv
    snapshot_date = file_name.replace("bronze_feature_attributes_", "").replace(".csv", "")

    # Read Bronze consistently as strings; type casting will be enforced in Silver
    df = spark.read.csv(file_path, header=True, inferSchema=False)

    row_count = df.count()
    column_count = len(df.columns)
    distinct_customer_count = df.select("customer_id").distinct().count()

    results.append({
        "snapshot_date": snapshot_date,
        "row_number": row_count,
        "column_number": column_count,
        "distinct_customer_id": distinct_customer_count,
        "customer_id_unique_within_snapshot": (
            row_count == distinct_customer_count
        )
    })

attributes_summary = pd.DataFrame(results).sort_values("snapshot_date")
attributes_summary

,snapshot_date,row_number,column_number,distinct_customer_id,customer_id_unique_within_snapshot
0,2023_01_01,530,6,530,True
1,2023_02_01,501,6,501,True
2,2023_03_01,506,6,506,True
3,2023_04_01,510,6,510,True
4,2023_05_01,521,6,521,True
5,2023_06_01,517,6,517,True
6,2023_07_01,471,6,471,True
7,2023_08_01,481,6,481,True
8,2023_09_01,454,6,454,True
9,2023_10_01,487,6,487,True


**Observations**: 6 source columns in every Bronze partition, `Customer_id` is unique in each partition. From here on, all 25 partitions are loaded as one table so each column can be profiled across the full dataset


### 2. Exploration  at union-level

In [75]:
# Union all partitions into one dataframe
attribute_dfs = [
    spark.read.csv(file_path, header=True, inferSchema=False)
    for file_path in attribute_files
]

attributes_df = attribute_dfs[0]
for partition_df in attribute_dfs[1:]:
    attributes_df = attributes_df.unionByName(partition_df)

print(f"Total rows: {attributes_df.count()}")
attributes_df.printSchema()
attributes_df.show(5, truncate=False)

Total rows: 12500
root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- snapshot_date: string (nullable = true)

+-----------+---------------+---+-----------+----------+-------------+
|Customer_ID|Name           |Age|SSN        |Occupation|snapshot_date|
+-----------+---------------+---+-----------+----------+-------------+
|CUS_0x1037 |Matthewm       |45 |230-22-9583|Accountant|2023-01-01   |
|CUS_0x1069 |Andreas Cremero|32 |761-27-5143|Accountant|2023-01-01   |
|CUS_0x114a |Valetkevitchu  |43 |133-89-5234|Developer |2023-01-01   |
|CUS_0x1184 |Cohenq         |49 |963-76-2464|Lawyer    |2023-01-01   |
|CUS_0x1297 |Edwardsz       |46 |#F%$D@*&8  |Manager   |2023-01-01   |
+-----------+---------------+---+-----------+----------+-------------+
only showing top 5 rows



#### 2.1. Check duplicates

In [76]:
# Number of rows
total_rows = attributes_df.count()

# Number of exact duplicate rows (excluding the first occurrence)
exact_duplicate_rows = (
    attributes_df.groupBy(attributes_df.columns)
    .count()
    .filter(F.col("count") > 1)
    .select(F.sum(F.col("count") - 1).alias("duplicate_rows"))
    .first()["duplicate_rows"] or 0
)

# Number of duplicated (snapshot_date, customer_id) pairs
duplicated_pairs = (
    attributes_df.groupBy("snapshot_date", "Customer_ID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_id = attributes_df.filter(F.col("Customer_ID").isNull()).count()
null_snapshot_date = attributes_df.filter(F.col("snapshot_date").isNull()).count()

print(f"Number of rows: {total_rows}")
print(f"Number of exact duplicated rows: {exact_duplicate_rows}")
print(f"Number of duplicated (snapshot_date, customer_id) pairs: {duplicated_pairs}")
print(f"Null Customer_ID: {null_customer_id}")
print(f"Null snapshot_date: {null_snapshot_date}")

Number of rows: 12500
Number of exact duplicated rows: 0
Number of duplicated (snapshot_date, customer_id) pairs: 0
Null Customer_ID: 0
Null snapshot_date: 0


#### 2.2. Scan generic missing values (null-like strings)

In [77]:
# Values treated as missing, regardless of letter case
missing_values = ("", "unknown", "n/a", "na", "nan", "null", "none")

missing_count_expressions = []

for column_name in attributes_df.columns:
    column_text = F.lower(
        F.trim(F.col(column_name).cast("string"))
    )
    is_missing = (
        F.col(column_name).isNull()
        | column_text.isin(*missing_values)
    )

    missing_count_expressions.append(
        F.sum(F.when(is_missing, 1).otherwise(0)).alias(column_name)
    )

attributes_df.select(*missing_count_expressions).show(truncate=False)

+-----------+----+---+---+----------+-------------+
|Customer_ID|Name|Age|SSN|Occupation|snapshot_date|
+-----------+----+---+---+----------+-------------+
|0          |0   |0  |0  |0         |0            |
+-----------+----+---+---+----------+-------------+



### 3. Clean `Age` column

#### 3.1. Inspect raw data

In [78]:
print("Age source datatype:")
print(attributes_df.schema["Age"])

Age source datatype:
StructField('Age', StringType(), True)


In [57]:
# Check rows that are not numeric
age_invalid = attributes_df.filter(
    ~F.col("Age").rlike(r"^\d+$")
)

age_invalid.select("Age").show(50, truncate=False)
print(f"Invalid Age rows: {age_invalid.count()}")

+-----+
|Age  |
+-----+
|41_  |
|-500 |
|2329_|
|38_  |
|39_  |
|18_  |
|28_  |
|15_  |
|45_  |
|26_  |
|31_  |
|-500 |
|42_  |
|23_  |
|48_  |
|45_  |
|-500 |
|29_  |
|-500 |
|-500 |
|-500 |
|43_  |
|32_  |
|34_  |
|21_  |
|35_  |
|-500 |
|31_  |
|37_  |
|43_  |
|43_  |
|-500 |
|21_  |
|43_  |
|27_  |
|25_  |
|45_  |
|7723_|
|41_  |
|-500 |
|-500 |
|14_  |
|38_  |
|51_  |
|43_  |
|44_  |
|35_  |
|-500 |
|55_  |
|35_  |
+-----+
only showing top 50 rows

Invalid Age rows: 741


**Observations:** `age` contains anomalies:
- Trailing underscores, e.g. `41_`, `2329_`  ==> Formatting issues
- Negative values, e.g: -500  ==> Invalid value

In [81]:
# Values containing an underscore
age_with_underscore = attributes_df.filter(
    F.col("Age").contains("_")
)

# Confirm the underscore pattern is only trailing underscores.
unexpected_underscore_pattern = attributes_df.filter(
    F.col("Age").contains("_")
    & ~F.col("Age").rlike(r"^-?\d+_+$")
)

print(f"Rows containing underscore: {age_with_underscore.count()}")
print(
    "Rows where underscore is NOT a simple trailing-suffix pattern: "
    f"{unexpected_underscore_pattern.count()}"
)

age_with_underscore.select("Age").distinct().orderBy("Age").show(10, truncate=False)

Rows containing underscore: 637
Rows where underscore is NOT a simple trailing-suffix pattern: 0
+-----+
|Age  |
+-----+
|1070_|
|1248_|
|14_  |
|15_  |
|16_  |
|17_  |
|18_  |
|19_  |
|20_  |
|21_  |
+-----+
only showing top 10 rows



#### 3.2. Handle formatting issue & cast to integer

In [85]:
age_profile_df = (
    attributes_df
    # Remove trailing underscore
    .withColumn(
        "age_normalized",
        F.regexp_replace(F.trim(F.col("Age")), r"_+$", "")
    )
    # Cast to integer
    .withColumn(
        "age_parsed",
        F.col("age_normalized").cast(IntegerType())
    )
)

# Verify that the formatting normalisation did not create unparseable ages.
age_profile_df.filter(
    F.col("Age").isNotNull() & F.col("age_parsed").isNull()
).select("Age", "age_normalized").show(50, truncate=False)

print(
    "Rows that could not be parsed after normalisation: "
    f"{age_profile_df.filter(F.col('Age').isNotNull() & F.col('age_parsed').isNull()).count()}"
)


print("Rows after normalisation and parsing:")
age_profile_df.select('Age', 'age_normalized', 'age_parsed').show()

+---+--------------+
|Age|age_normalized|
+---+--------------+
+---+--------------+

Rows that could not be parsed after normalisation: 0
Rows after normalisation and parsing:
+---+--------------+----------+
|Age|age_normalized|age_parsed|
+---+--------------+----------+
| 45|            45|        45|
| 32|            32|        32|
| 43|            43|        43|
| 49|            49|        49|
| 46|            46|        46|
| 35|            35|        35|
| 27|            27|        27|
| 47|            47|        47|
| 15|            15|        15|
|41_|            41|        41|
| 52|            52|        52|
| 19|            19|        19|
| 31|            31|        31|
| 29|            29|        29|
| 36|            36|        36|
| 20|            20|        20|
| 31|            31|        31|
| 27|            27|        27|
| 35|            35|        35|
| 46|            46|        46|
+---+--------------+----------+
only showing top 20 rows



#### 3.3. Inspect range of `Age`

**Business sense:**
- Negative values (e.g: -500) and ages above 100 are implausible ==> Convert to Null
- Values < 18 does not make sense in business & legal context ==> Retain as observed in Silver and flag for Gold 

In [88]:
# Overall range checks
age_counts = age_profile_df.select(
    F.sum(F.when(F.col("age_parsed") < 0, 1).otherwise(0))
        .alias("count_age_below_0"),
    F.sum(F.when(F.col("age_parsed") < 18, 1).otherwise(0))
        .alias("count_age_below_18"),
    F.sum(F.when(F.col("age_parsed") > 100, 1).otherwise(0))
        .alias("count_age_above_100")
)

age_counts.show()


# Inspect frequencies outside the expected 0-100 quality range
(
    age_profile_df
    .filter(
        (F.col("age_parsed") < 0)
        | (F.col("age_parsed") > 100)
        | (F.col("age_parsed") < 18)
    )
    .groupBy("age_parsed")
    .count()
    .orderBy("age_parsed")
    .show(150, truncate=False)
)

+-----------------+------------------+-------------------+
|count_age_below_0|count_age_below_18|count_age_above_100|
+-----------------+------------------+-------------------+
|              104|               773|                215|
+-----------------+------------------+-------------------+

+----------+-----+
|age_parsed|count|
+----------+-----+
|-500      |104  |
|14        |87   |
|15        |195  |
|16        |189  |
|17        |198  |
|169       |1    |
|203       |1    |
|210       |1    |
|236       |1    |
|305       |1    |
|306       |1    |
|325       |1    |
|335       |1    |
|344       |1    |
|395       |1    |
|448       |1    |
|506       |1    |
|509       |1    |
|581       |1    |
|598       |1    |
|655       |2    |
|813       |1    |
|831       |1    |
|886       |1    |
|925       |1    |
|1004      |1    |
|1022      |1    |
|1066      |1    |
|1070      |1    |
|1087      |1    |
|1094      |1    |
|1149      |1    |
|1203      |1    |
|1220      |1    |
|

In [89]:
# Apply the Silver cleaning rule for Age (replace value <0 and >100 with null)
age_clean_df = (
    age_profile_df
    .withColumn(
        "age_clean",
        F.when(
            (F.col("age_parsed") < 0)
            | (F.col("age_parsed") > 100),
            F.lit(None).cast(IntegerType())
        ).otherwise(F.col("age_parsed"))
    )
)

# Validate the rule
remaining_invalid_age = age_clean_df.filter(
    (F.col("age_clean") < 0) | (F.col("age_clean") > 100)
).count()

print(f"Remaining impossible age values: {remaining_invalid_age}")
print(
    "Age values converted to NULL: "
    f"{age_clean_df.filter(F.col('age_clean').isNull()).count()}"
)

Remaining impossible age values: 0
Age values converted to NULL: 319


### 4. Clean `Occupation` column

#### 4.1. Inspect raw data & profile categories

In [63]:
attributes_df.groupBy("Occupation").count().orderBy(F.desc("count")).show()

+-------------+-----+
|   Occupation|count|
+-------------+-----+
|      _______|  880|
|       Lawyer|  828|
|    Architect|  795|
|     Engineer|  793|
|   Accountant|  791|
|    Scientist|  789|
|      Teacher|  782|
|Media_Manager|  780|
|    Developer|  780|
|     Mechanic|  780|
| Entrepreneur|  776|
|   Journalist|  761|
|       Doctor|  760|
|     Musician|  741|
|      Manager|  736|
|       Writer|  728|
+-------------+-----+



#### 4.2. Handle placeholder

Convert placeholder `_______` to Null, retain valid category names as they are

In [93]:
occupation_clean_df = (
    age_clean_df
    .withColumn(
        "occupation_clean",
        F.when(
            F.trim(F.col("Occupation")) == "_______",
            F.lit(None).cast(StringType())
        ).otherwise(F.trim(F.col("Occupation")))
    )
)

print(
    "Occupation NULL count after cleaning: "
    f"{occupation_clean_df.filter(F.col('occupation_clean').isNull()).count()}"
)

(
    occupation_clean_df
    .groupBy("occupation_clean")
    .count()
    .orderBy(F.desc("count"))
    .show(100, truncate=False)
)

Occupation NULL count after cleaning: 880
+----------------+-----+
|occupation_clean|count|
+----------------+-----+
|NULL            |880  |
|Lawyer          |828  |
|Architect       |795  |
|Engineer        |793  |
|Accountant      |791  |
|Scientist       |789  |
|Teacher         |782  |
|Media_Manager   |780  |
|Developer       |780  |
|Mechanic        |780  |
|Entrepreneur    |776  |
|Journalist      |761  |
|Doctor          |760  |
|Musician        |741  |
|Manager         |736  |
|Writer          |728  |
+----------------+-----+



### Build Silver `features_attributes` table

| Source column | Silver decision | Reason |
|---|---|---|
| `Customer_ID` | Keep → `customer_id` | Entity key for joins |
| `Name` | Drop | PII; not required for ML features |
| `Age` | Keep cleaned → `age` | Candidate model feature |
| `SSN` | Drop | Sensitive PII; not required for ML features |
| `Occupation` | Keep cleaned → `occupation` | Candidate model feature |
| `snapshot_date` | Keep as `DateType` | Temporal alignment / leakage control |

In [97]:
silver_attributes_df = (
    occupation_clean_df
    .withColumn(
        "customer_id",
        F.trim(F.col("Customer_ID")).cast(StringType())
    )
    .withColumn(
        "snapshot_date_clean",
        F.to_date(F.col("snapshot_date"), "yyyy-MM-dd")
    )
    .select(
        "customer_id",
        F.col("age_clean").alias("age"),
        F.col("occupation_clean").alias("occupation"),
        F.col("snapshot_date_clean").alias("snapshot_date")
    )
)

silver_attributes_df.printSchema()
silver_attributes_df.show(20, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- occupation: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

+-----------+---+-------------+-------------+
|customer_id|age|occupation   |snapshot_date|
+-----------+---+-------------+-------------+
|CUS_0x1037 |45 |Accountant   |2023-01-01   |
|CUS_0x1069 |32 |Accountant   |2023-01-01   |
|CUS_0x114a |43 |Developer    |2023-01-01   |
|CUS_0x1184 |49 |Lawyer       |2023-01-01   |
|CUS_0x1297 |46 |Manager      |2023-01-01   |
|CUS_0x12fb |35 |Doctor       |2023-01-01   |
|CUS_0x1325 |27 |Accountant   |2023-01-01   |
|CUS_0x1341 |47 |Mechanic     |2023-01-01   |
|CUS_0x1375 |15 |Lawyer       |2023-01-01   |
|CUS_0x13a8 |41 |Journalist   |2023-01-01   |
|CUS_0x13ef |52 |Media_Manager|2023-01-01   |
|CUS_0x1440 |19 |Teacher      |2023-01-01   |
|CUS_0x1443 |31 |Lawyer       |2023-01-01   |
|CUS_0x145a |29 |Lawyer       |2023-01-01   |
|CUS_0x1492 |36 |Teacher      |2023-01-01   |
|C

In [100]:
# Validation against Bronze to ensure no rows are accidentally removed

bronze_row_count = attributes_df.count()
silver_row_count = silver_attributes_df.count()

silver_null_summary = silver_attributes_df.select(
    *[
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in silver_attributes_df.columns
    ]
)

print(f"Bronze rows: {bronze_row_count}")
print(f"Silver rows: {silver_row_count}")
print(f"Row count preserved: {bronze_row_count == silver_row_count}")
print("Null counts after Silver cleaning:")
silver_null_summary.show(truncate=False)

Bronze rows: 12500
Silver rows: 12500
Row count preserved: True
Null counts after Silver cleaning:
+-----------+---+----------+-------------+
|customer_id|age|occupation|snapshot_date|
+-----------+---+----------+-------------+
|0          |319|880       |0            |
+-----------+---+----------+-------------+



## Build Silver tables for `feature_attributes`

In [7]:
bronze_directory = "datamart/bronze/attributes"
silver_directory = "datamart/silver/attributes"

for snapshot_date in date_str_lst:
    utils.data_processing_silver_table.process_feature_attributes_silver_table(
        snapshot_date,
        bronze_directory,
        silver_directory,
        spark
    )

loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_01_01.csv row count: 530


saved to: datamart/silver/attributes/silver_feature_attributes_2023_01_01.parquet row count: 530
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_02_01.csv row count: 501
saved to: datamart/silver/attributes/silver_feature_attributes_2023_02_01.parquet row count: 501
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_03_01.csv row count: 506
saved to: datamart/silver/attributes/silver_feature_attributes_2023_03_01.parquet row count: 506
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_04_01.csv row count: 510
saved to: datamart/silver/attributes/silver_feature_attributes_2023_04_01.parquet row count: 510
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_05_01.csv row count: 521
saved to: datamart/silver/attributes/silver_feature_attributes_2023_05_01.parquet row count: 521
loaded from: datamart/bronze/attributes/bronze_feature_attributes_2023_06_01.csv row count: 517
saved to: datamart/silver/attribute